# Przykład 1: Q-learning w korytarzu 1D

Agent porusza się po korytarzu i szuka pola celu:

```
[A][ ][ ][ ][ ][G]
```

Stan to pozycja, a akcje to ruch w lewo lub w prawo. Za wejście na cel agent dostaje `+10`, a za każdy inny krok `-1`. Epizod kończy się po osiągnięciu celu lub limitu kroków.

Q-learning aktualizuje wartość wybranej akcji:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$$

W stanie terminalnym pomijamy składnik przyszłej wartości. Uruchom komórki po kolei i obserwuj przebieg uczenia oraz wyuczoną politykę.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Parametry

In [ ]:
# --- środowisko ---
N_CELLS = 6            # liczba pól korytarza [-]; cel to zawsze ostatnie pole (N_CELLS - 1)
RANDOM_START = True    # True: losowy start spośród pól innych niż cel, False: zawsze pole 0
MAX_STEPS = 100        # limit kroków w jednym epizodzie [kroki]

# --- nagrody ---
REWARD_GOAL = 10.0     # nagroda za wejście na cel [-]
REWARD_STEP = -1.0     # nagroda (kara) za każdy pozostały krok [-]

# --- Q-learning ---
N_EPISODES = 300       # liczba epizodów uczenia [-]
ALPHA = 0.5            # współczynnik uczenia, (0, 1] [-]
GAMMA = 0.9            # współczynnik dyskontowania, (0, 1] [-]

# --- strategia epsilon-zachłanna ---
EPSILON_START = 1.0    # epsilon w pierwszym epizodzie, [0, 1] [-]
EPSILON_END = 0.05     # epsilon w ostatnim epizodzie, [0, 1] [-]
EPSILON_MODE = "linear"  # "const" (stałe EPSILON_START), "linear" lub "exp" (redukcja)

SEED = 0               # ziarno generatora losowego (powtarzalność wyników)

## Środowisko

Środowisko opisuje ruch i nagrody; agent otrzymuje tylko stan, nagrodę i informację o końcu epizodu. Próba ruchu poza krawędź korytarza pozostawia agenta w miejscu i nadal nalicza zwykłą karę za krok.

In [ ]:
ACTIONS = ["lewo", "prawo"]          # nazwy akcji; indeks akcji = pozycja na liście
ACTION_DELTA = [-1, +1]              # zmiana pozycji dla każdej akcji [pola]
N_ACTIONS = len(ACTIONS)
GOAL = N_CELLS - 1                   # indeks pola celu


class CorridorEnv:
    """Korytarz 1D. Obserwacją jest indeks pola, na którym stoi agent."""

    def __init__(self, n_cells, random_start):
        self.n_cells = n_cells
        self.random_start = random_start
        self.position = 0            # aktualne pole agenta [indeks pola]

    def reset(self, rng):
        """Zaczyna nowy epizod i zwraca obserwację początkową."""
        if self.random_start:
            self.position = int(rng.integers(0, self.n_cells - 1))  # bez pola celu
        else:
            self.position = 0
        return self.position

    def step(self, action):
        """Wykonuje akcję. Zwraca (obserwacja, nagroda, czy_terminalny)."""
        if not 0 <= action < N_ACTIONS:
            raise ValueError(f"Niepoprawny numer akcji: {action}")
        # próba wyjścia poza korytarz zostawia agenta na skrajnym polu
        self.position = int(np.clip(self.position + ACTION_DELTA[action], 0, self.n_cells - 1))
        if self.position == GOAL:
            return self.position, REWARD_GOAL, True
        return self.position, REWARD_STEP, False


def draw_corridor(position, n_cells=None):
    """Zwraca tekstową wizualizację korytarza, np. `[ ][A][ ][ ][ ][G]`."""
    n_cells = n_cells or N_CELLS
    cells = []
    for i in range(n_cells):
        if i == position:
            cells.append("[A]")
        elif i == n_cells - 1:
            cells.append("[G]")
        else:
            cells.append("[ ]")
    return "".join(cells)


for p in range(N_CELLS):
    print(f"pole {p}: {draw_corridor(p)}")

## Stan

Stanem jest pozycja agenta, więc nie trzeba jej dodatkowo kodować.

In [ ]:
def encode_state(observation):
    """Zamienia obserwację na indeks wiersza tablicy Q. Tu: bez zmian."""
    return int(observation)

## Q-learning

Agent wybiera akcje metodą epsilon-zachłanną i aktualizuje tablicę Q po każdym kroku.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    """Strategia epsilon-zachłanna.

    Z prawdopodobieństwem `epsilon` zwraca akcję losową (eksploracja), w przeciwnym razie
    akcję o największej wartości Q (eksploatacja). Remisy (np. na początku, gdy cała tablica
    to zera) są rozstrzygane losowo — inaczej agent zawsze brałby akcję 0.
    """
    n_actions = q_table.shape[1]
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    best = np.flatnonzero(q_table[state] == q_table[state].max())
    return int(rng.choice(best))


def update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma):
    """Jedna aktualizacja Q-learningu dla przejścia (s, a, r, s')."""
    if terminal:
        target = reward                                           # brak następnego kroku
    else:
        target = reward + gamma * np.max(q_table[next_state])     # r + gamma * max Q(s', a')
    td_error = target - q_table[state, action]                    # błąd różnicy czasowej
    q_table[state, action] += alpha * td_error


def epsilon_for_episode(episode, n_episodes, eps_start, eps_end, mode):
    """Wartość epsilon w danym epizodzie (numerowanym od 0)."""
    if mode == "const":
        return eps_start
    progress = episode / max(1, n_episodes - 1)                   # 0 … 1
    if mode == "linear":
        return eps_start + (eps_end - eps_start) * progress
    if mode == "exp":
        return eps_start * (eps_end / eps_start) ** progress
    raise ValueError(f"Nieznany tryb epsilon: {mode}")


def train(env, n_states, n_actions, n_episodes, alpha, gamma,
          eps_start, eps_end, eps_mode, max_steps, seed):
    """Uczy agenta. Zwraca tablicę Q i historię uczenia (po jednej wartości na epizod)."""
    rng = np.random.default_rng(seed)
    q_table = np.zeros((n_states, n_actions))
    history = {"return": [], "steps": [], "success": [], "epsilon": []}

    for episode in range(n_episodes):
        epsilon = epsilon_for_episode(episode, n_episodes, eps_start, eps_end, eps_mode)
        state = encode_state(env.reset(rng))
        total_reward, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = choose_action(q_table, state, epsilon, rng)
            observation, reward, terminal = env.step(action)
            next_state = encode_state(observation)
            # Aktualizacja dopiero PO poznaniu skutków akcji (r i s').
            # Limit kroków NIE jest stanem terminalnym, więc nadal liczymy max Q(s', .).
            update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma)
            state = next_state
            total_reward += reward
            steps += 1

        history["return"].append(total_reward)
        history["steps"].append(steps)
        history["success"].append(terminal)
        history["epsilon"].append(epsilon)
    return q_table, history

## Początkowa tablica Q

Wiersze odpowiadają pozycjom, a kolumny akcjom. Początkowe wartości wynoszą zero.

In [ ]:
env = CorridorEnv(N_CELLS, RANDOM_START)


def print_q_table(q_table):
    print("pole |  Q(lewo)  Q(prawo) | najlepsza akcja")
    for s, row in enumerate(q_table):
        best = "— (cel)" if s == GOAL else ACTIONS[int(np.argmax(row))]
        print(f"{s:4d} | {row[0]:8.2f} {row[1]:9.2f} | {best}")


print_q_table(np.zeros((N_CELLS, N_ACTIONS)))

## Trening

In [ ]:
q_table, history = train(env, N_CELLS, N_ACTIONS, N_EPISODES, ALPHA, GAMMA,
                         EPSILON_START, EPSILON_END, EPSILON_MODE, MAX_STEPS, SEED)
print(f"Trening zakończony: {N_EPISODES} epizodów, "
      f"skuteczność w ostatnich 50 epizodach (z eksploracją): "
      f"{np.mean(history['success'][-50:]):.0%}")

### Obserwuj
- Czy suma nagród rośnie, a liczba kroków maleje?
- Malejące `epsilon` oznacza coraz rzadsze losowe ruchy.

In [ ]:
def moving_average(values, window=20):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["return"], alpha=0.3, label="pojedynczy epizod")
ma = moving_average(history["return"])
axes[0].plot(np.arange(len(history["return"]) - len(ma), len(history["return"])), ma, label="średnia krocząca (20)")
axes[0].set(xlabel="numer epizodu [-]", ylabel="suma nagród [-]", title="Suma nagród w epizodzie")
axes[0].legend()

axes[1].plot(history["steps"], alpha=0.3, label="pojedynczy epizod")
ma = moving_average(history["steps"])
axes[1].plot(np.arange(len(history["steps"]) - len(ma), len(history["steps"])), ma, label="średnia krocząca (20)")
axes[1].set(xlabel="numer epizodu [-]", ylabel="liczba kroków [kroki]", title="Długość epizodu")
axes[1].legend()

axes[2].plot(history["epsilon"], color="tab:red", label="epsilon")
axes[2].set(xlabel="numer epizodu [-]", ylabel="epsilon [-]", title="Eksploracja")
axes[2].legend()
plt.tight_layout()
plt.show()

## Wyuczona polityka

Dla każdego stanu wybierana jest akcja o najwyższej wartości Q. Oczekiwany ruch prowadzi do celu.

In [ ]:
print_q_table(q_table)

policy = np.argmax(q_table, axis=1)
print("\nPolityka:")
print("".join(f"[{'<' if policy[s] == 0 else '>'}]" if s != GOAL else "[G]" for s in range(N_CELLS)))

fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(q_table.T, cmap="viridis", aspect="auto")
ax.set_xticks(range(N_CELLS))
ax.set_yticks(range(N_ACTIONS))
ax.set_yticklabels(ACTIONS)
ax.set(xlabel="stan: numer pola [-]", ylabel="akcja", title="Tablica Q")
fig.colorbar(im, label="Q [-]")
plt.show()

## Demonstracja

Agent przechodzi trasę od wybranej pozycji, korzystając z wyuczonej polityki bez losowych ruchów.

In [ ]:
DEMO_START = 0         # pole startowe demonstracji [indeks pola]

demo_env = CorridorEnv(N_CELLS, random_start=False)
demo_env.reset(np.random.default_rng(SEED))
demo_env.position = DEMO_START
state = encode_state(demo_env.position)
rng_demo = np.random.default_rng(SEED)

print(f"start    {draw_corridor(demo_env.position)}")
for t in range(1, MAX_STEPS + 1):
    action = choose_action(q_table, state, 0.0, rng_demo)
    observation, reward, terminal = demo_env.step(action)
    state = encode_state(observation)
    print(f"krok {t:2d}  {draw_corridor(observation)}  akcja={ACTIONS[action]:5s} r={reward:+.0f}")
    if terminal:
        print("Cel osiągnięty.")
        break
else:
    print("Limit kroków — cel nieosiągnięty.")